# LC #121 — Best Time to Buy and Sell Stock
**Category:** Sliding Window / Two Pointers | **Difficulty:** Easy | **Day 2**

---

<div style="padding:15px;border-left:8px solid #11998e;background:#e8faf5;border-radius:4px;">
<strong>The Problem:</strong> Given array <code>prices</code> where <code>prices[i]</code> is the stock price on day i,
return the maximum profit from one buy and one sell. Return 0 if no profit is possible.
</div>

**Examples:**
```
[7, 1, 5, 3, 6, 4] → 5   (buy at 1, sell at 6)
[7, 6, 4, 3, 1]    → 0   (prices only fall — no profit)
[1, 2]             → 1
```

**Core Insight:** For each potential sell day, the best buy is the *minimum price to its left*. Track the running minimum as you scan left-to-right. At each price, the potential profit is `current - min_so_far`.

## Mental Models

**1. The Running Minimum**
As you scan left-to-right, maintain the lowest price seen so far. At each new price, calculate potential profit = `current - min_so_far`. Update the running max profit if higher.

**2. Sliding Window View**
The "window" implicitly spans from `min_price_index` to `current_index`. You never explicitly track the left pointer — the minimum tracks it for you.

**3. Time Series Analogy**
Identical pattern to: "For each day's CPU reading, what's the maximum improvement over any prior baseline?" Rolling maximum of (current - historical_min) is the same calculation.

In [ ]:
# Brute Force — O(n²) time, O(1) space
# Check every buy-sell pair.

def maxProfit_brute(prices):
    max_profit = 0
    for i in range(len(prices)):
        for j in range(i + 1, len(prices)):
            max_profit = max(max_profit, prices[j] - prices[i])
    return max_profit

# Test
print(maxProfit_brute([7, 1, 5, 3, 6, 4]))   # 5
print(maxProfit_brute([7, 6, 4, 3, 1]))       # 0
print(maxProfit_brute([1, 2]))                 # 1

## Step-by-Step: One-Pass Approach

Trace through `[7, 1, 5, 3, 6, 4]`:

| Day | Price | min_price | profit = price - min | max_profit |
|-----|-------|-----------|---------------------|------------|
| 0 | 7 | 7 | 0 | 0 |
| 1 | 1 | **1** | 0 | 0 |
| 2 | 5 | 1 | **4** | 4 |
| 3 | 3 | 1 | 2 | 4 |
| 4 | 6 | 1 | **5** | **5** |
| 5 | 4 | 1 | 3 | 5 |

At day 4 we find the max: buy at 1 (day 1), sell at 6 (day 4) = profit 5.

In [ ]:
# Optimal — O(n) time, O(1) space
# One pass: track min price and max profit simultaneously.

def maxProfit(prices: list[int]) -> int:
    min_price  = float('inf')
    max_profit = 0

    for price in prices:
        if price < min_price:
            min_price = price
        elif price - min_price > max_profit:
            max_profit = price - min_price

    return max_profit

# Test
print(maxProfit([7, 1, 5, 3, 6, 4]))   # 5
print(maxProfit([7, 6, 4, 3, 1]))       # 0
print(maxProfit([1, 2]))                # 1
print(maxProfit([2, 4, 1]))             # 2

## Complexity Analysis

| Approach | Time | Space | Notes |
|----------|------|-------|-------|
| Brute Force (nested) | O(n²) | O(1) | Check every pair |
| **One-Pass (Optimal)** | **O(n)** | **O(1)** | Track running min |

**Why O(1) space?** Only two variables: `min_price` and `max_profit`. No data structure needed.

**Key constraint:** You must buy before you sell. Scanning left-to-right respects this — `min_price` is always from a prior day.

## Interview Q&A

**Q: Why is this categorized as sliding window?**
A: The window is the range [buy_day, sell_day]. The left boundary is the running minimum index (implicitly). The right boundary is the current day. Unlike explicit two-pointer sliding windows, we don't need to track both pointers — the min does it.

**Q: How would you modify this for "buy and sell multiple times"?**
A: LC #122 — Greedy: add all positive day-over-day differences. `sum(max(0, prices[i] - prices[i-1]) for i in range(1, len(prices)))`.

**Q: How would you modify for "at most 2 transactions"?**
A: LC #123 — Track four states with DP: after first buy, after first sell, after second buy, after second sell.

**Q: What's the data engineering analog of this problem?**
A: "Find the maximum metric improvement over any prior baseline." Same algorithm — track rolling minimum, compute delta at each point.

## The Citi Angle

**Direct capacity planning analog:** "What is the maximum performance headroom we ever had compared to our worst baseline in the past 30 days?"

```python
# Real pattern: rolling max improvement from historical low
readings = [72, 68, 75, 71, 80, 65, 88]   # daily avg CPU readings
baseline = float('inf')
max_headroom = 0

for cpu in readings:
    baseline = min(baseline, cpu)
    headroom = cpu - baseline   # how much headroom above historical low
    max_headroom = max(max_headroom, headroom)

print(f"Max usage swing from baseline: {max_headroom}%")
```

**Interview tie-in:** "The buy-sell pattern is identical to finding the maximum metric delta from a rolling baseline — a capacity planning calculation I did regularly at Citi to understand usage volatility."

## Summary

| | Value |
|--|--|
| **Pattern** | Sliding Window / Running Minimum |
| **Time** | O(n) |
| **Space** | O(1) |
| **Key insight** | `profit = current_price - min_price_so_far` |
| **Say in interview** | "One pass: track running minimum and max profit. O(n) time, O(1) space." |

**The 3-line core:**
```python
for price in prices:
    min_price = min(min_price, price)
    max_profit = max(max_profit, price - min_price)
```